# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DataType
from pyspark.sql.functions import lit, col, trim

# Read from Bronze Table

In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2")

# Silver Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name )))

## Normalize Maintenance Flag to Boolean

In [0]:
df = df.withColumn(
    "maintenance",
    F.when(F.upper(col("maintenance")) == "YES", F.lit(True))
    .when(F.upper(col("maintenance")) == "NO", F.lit(False))
    .otherwise(None)
)

## Renaming Columns

In [0]:
RENAME_MAP = {
    "id" : "category_id",
    "cat" : "category",
    "subcat" : "subcategory",
    "maintenance" : "maintenance"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Write to Silver Table

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_product_category")

## Sanity Check of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_product_category LIMIT 10